# Build a Simple Neural Network

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import tensorflow as tf 

## Define some functions

In [2]:
def relu(x):
    # relu function
    return np.maximum(x, 0)

In [3]:
# test relu
relu(np.array([[-2, 3],
              [5, -3]]))

array([[0, 3],
       [5, 0]])

In [4]:
def cost_J(u, y):
    # MSE cost function: 1 / 2 * ||u-y||^2
    return 0.5*(np.sum((u - y)**2))

In [5]:
# test cost
cost_J(np.array([1,2]), np.array([2,3]))

np.float64(1.0)

## train the weights
The XOR (Exclusive OR) function is a logical operation that returns TRUE if exactly one of the inputs is true, but FALSE if both are true or both are false

In [6]:
# XOR function
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0]).reshape([-1, 1])
# "-1" above means automatically compute the other dimension
print(X)
print(y)

[[0 0]
 [0 1]
 [1 0]
 [1 1]]
[[0]
 [1]
 [1]
 [0]]


In [7]:
n, p = X.shape

# size (number of nodes) of hidden layer
p1 = 5
lr = 1e-1
max_iter = 30

# initialization
np.random.seed(2)
W1 = np.random.randn(p, p1) # shape is p by p1
W2 = np.random.randn(p1, 1)

for i in range(max_iter):
    # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
    u1 = X@W1
    h = relu(u1)
    u2 = h@W2

    # compute cost
    J = cost_J(u2, y)
    #print(J)

    # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
    du2 = u2 - y
    dW2 = (h.T)@du2
    dh = du2@W2.T
    # du1 = dh*r'(u1): if u1(i,j) > 0, then r'(u1(i,j))=1; if u1(i,j) <= 0, then r'(u1(i,j))=0
    du1 = dh*relu(np.sign(u1))
    dW1 = X.T@du1
    
    # gradient descent
    W1 = W1 - lr * dW1
    W2 = W2 - lr * dW2
print(u2)

[[0.        ]
 [0.99999995]
 [0.96052743]
 [0.07128983]]


## Try train this with linear regression: fail

In [8]:
lm = LinearRegression().fit(X,y)
print(lm.predict(X))
print(y)

[[0.5]
 [0.5]
 [0.5]
 [0.5]]
[[0]
 [1]
 [1]
 [0]]


## write a function to do simple NN

In [9]:
def simple_NN(X, y, seed = 2, lr = 1e-1, hidden_size = 5, max_iter = 30):
    '''
    X, y should be numpy arrays
    '''
    n, p = X.shape
    # reshape y to a column
    y = y.reshape([-1, 1])

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # initialization
    np.random.seed(seed)
    W1 = np.random.randn(p, p1) 
    W2 = np.random.randn(p1, 1)
    
    for i in range(max_iter):
        # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
        u1 = X@W1
        h = relu(u1)
        u2 = h@W2 # u2 is supposed to be approx y
    
        # compute cost
        J = cost_J(u2, y)
        #print(J)
    
        # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
        du2 = u2 - y
        dW2 = (h.T)@du2
        dh = du2@W2.T
        # du1 = dh*r'(u1): 
        # if u1(i,j) > 0, then r'(u1)(i,j)=1; if u1(i,j) <= 0, then r'(u1)(i,j)=0
        du1 = dh*relu(np.sign(u1)) # point-wise multiplication
        dW1 = X.T@du1
        
        # gradient descent
        W1 = W1 - lr * dW1
        W2 = W2 - lr * dW2
    return W1, W2, J, u2

In [10]:
# works on XOR
simple_NN(X,y)

(array([[-0.41675785, -0.43816504, -2.1361961 ,  1.24168812, -1.79343559],
        [-0.84174737,  0.43270045, -1.24528809, -1.15730864, -0.90900761]]),
 array([[0.55145404],
        [2.31106759],
        [0.04153939],
        [0.77629113],
        [0.53905832]]),
 np.float64(0.003320162025686291),
 array([[0.        ],
        [0.99999995],
        [0.96052743],
        [0.07128983]]))

## Let's try to use our NN to do sale price prediction

In [11]:
# data prep
ames = pd.read_csv("../data/ames.csv")
ames_num = ames.select_dtypes(exclude=['object'])
yh = ames['Sale_Price']
Xh = ames_num.iloc[:,:-3] # selecting all but the last 3 columns
X_train, X_test, y_train, y_test = train_test_split(Xh, yh, test_size = 0.25, random_state = 45)
X_train_n = X_train.to_numpy()
y_train_n = y_train.to_numpy()
X_test_n = X_test.to_numpy()
y_test_n = y_test.to_numpy()

### This is the result of linear regression

In [12]:
lm_h = LinearRegression().fit(X_train,y_train)

In [13]:
# use a data frame for output
results = pd.DataFrame({'training_RMSE':[],
                        'testing_RMSE':[]})
results.loc['linear',:] = [root_mean_squared_error(lm_h.predict(X_train), y_train.to_numpy()),
                           root_mean_squared_error(lm_h.predict(X_test), y_test.to_numpy())]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308


### Use Simple NN

In [16]:
p1

5

In [24]:
W1, W2, J, u2 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 1e-14, max_iter = int(1.5e5))
root_mean_squared_error(u2, y_train_n)

39530.607484929045

### How to use W1, W2 above to predict sale price for the testing data?

In [22]:
def simple_NN_pred(X_new, W1_trained, W2_trained):
    u1 = X_new@W1_trained
    h1 = relu(u1)
    u2 = h1@W2_trained
    return u2

In [25]:
root_mean_squared_error(simple_NN_pred(X_test_n, W1, W2), y_test_n)

42820.30917811267

In [32]:
results.loc['simpleNN',:] = [root_mean_squared_error(u2, y_train_n),
                             root_mean_squared_error(simple_NN_pred(X_test_n, W1, W2), y_test_n)]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308
simpleNN,39530.607485,42820.309178


## Standardizing can help with starting with a more stable learning rate
And usually it will decrease the max iterations needed

In [47]:
X_train

,Lot_Frontage,Lot_Area,Year_Built,Year_Remod_Add,Mas_Vnr_Area,BsmtFin_SF_1,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,First_Flr_SF,...,Garage_Area,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,Three_season_porch,Screen_Porch,Pool_Area,Misc_Val,Mo_Sold,Year_Sold
1233,90,14670,1966,1999,410,2,0,529,1104,1104,...,480,0,230,0,0,0,0,0,8,2008
1563,0,23730,1996,1997,668,3,0,1140,1840,2032,...,786,0,46,192,0,0,0,0,9,2008
847,67,9808,2002,2002,110,3,0,785,1573,1573,...,544,0,72,0,0,0,0,0,3,2009
263,80,10389,2003,2003,320,3,0,296,1978,1978,...,850,188,25,0,0,0,0,0,5,2010
1289,60,4800,1940,1950,0,7,0,672,672,672,...,256,0,0,0,0,0,0,0,8,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,71,8520,1952,1952,0,6,0,403,910,910,...,720,0,0,0,0,0,0,0,6,2010
607,115,10500,1964,1964,144,4,0,294,1292,1292,...,520,0,32,0,0,92,0,0,7,2009
1916,73,8760,1957,1999,0,1,0,63,936,936,...,315,0,0,0,219,0,0,0,9,2007
1667,61,9758,1971,1971,0,2,287,251,950,950,...,280,0,0,0,0,0,0,0,7,2007


In [30]:
X_train_transformed = (X_train - X_train.mean())/X_train.std()
X_train_transformed_n = X_train_transformed.to_numpy()
X_train_transformed

,Lot_Frontage,Lot_Area,Year_Built,Year_Remod_Add,Mas_Vnr_Area,BsmtFin_SF_1,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,First_Flr_SF,...,Garage_Area,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,Three_season_porch,Screen_Porch,Pool_Area,Misc_Val,Mo_Sold,Year_Sold
1233,0.963568,0.567468,-0.168453,0.717130,1.739483,-0.988028,-0.293365,-0.073111,0.122237,-0.139220,...,0.033813,-0.729390,2.723792,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,0.670326,0.151421
1563,-1.708423,1.708124,0.824634,0.621736,3.188653,-0.539629,-0.293365,1.326096,1.804396,2.281215,...,1.466896,-0.729390,-0.024262,2.522063,-0.095348,-0.297174,-0.059713,-0.087882,1.040166,0.151421
847,0.280726,-0.044659,1.023252,0.860222,0.054403,-0.539629,-0.293365,0.513136,1.194157,1.084039,...,0.333542,-0.729390,0.364050,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,-1.178877,0.910948
263,0.666680,0.028489,1.056355,0.907919,1.233959,-0.539629,-0.293365,-0.606687,2.119801,2.140371,...,1.766626,0.749389,-0.337898,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,-0.439196,1.670475
1289,0.072904,-0.675168,-1.029128,-1.620034,-0.563460,1.253964,-0.293365,0.254363,-0.865117,-1.265975,...,-1.015242,-0.729390,-0.711275,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,0.670326,0.151421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,0.399481,-0.206819,-0.631893,-1.524639,-0.563460,0.805566,-0.293365,-0.361654,-0.321158,-0.645216,...,1.157799,-0.729390,-0.711275,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,-0.069356,1.670475
607,1.705788,0.042464,-0.234658,-0.952273,0.245379,-0.091231,-0.293365,-0.611267,0.551919,0.351127,...,0.221144,-0.729390,-0.233353,-0.357034,-0.095348,1.354714,-0.059713,-0.087882,0.300485,0.910948
1916,0.458859,-0.176603,-0.466379,0.717130,-0.563460,-1.436426,-0.293365,-1.140264,-0.261734,-0.577403,...,-0.738928,-0.729390,-0.711275,-0.357034,8.594970,-0.297174,-0.059713,-0.087882,1.040166,-0.608105
1667,0.102593,-0.050954,-0.002938,-0.618392,-0.563460,-0.988028,1.337491,-0.709739,-0.229736,-0.540887,...,-0.902843,-0.729390,-0.711275,-0.357034,-0.095348,-0.297174,-0.059713,-0.087882,0.300485,-0.608105


In [31]:
# apply the same transformation to X_test
X_test_transformed = (X_test - X_train.mean())/X_train.std()
X_test_transformed_n = X_test_transformed.to_numpy()

In [48]:
W1_s, W2_s, J, u2 = simple_NN(X_train_transformed_n, y_train_n, hidden_size = p1, seed = 2, lr = 1e-9, max_iter = int(5e3))
root_mean_squared_error(u2, y_train_n)

29845.31104665943

In [49]:
results.loc['simpleNN_std',:] = [root_mean_squared_error(u2, y_train_n),
                                 root_mean_squared_error(simple_NN_pred(X_test_transformed_n, W1_s, W2_s), y_test_n)]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308
simpleNN,39530.607485,42820.309178
simpleNN_std,29845.311047,46228.593339


## Try tensorflow
```
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate = 2e-1,
    decay_steps = 20,
    decay_rate = 0.8)
opt = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

p = X_train.shape[1]
model_tf = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape = (p, )),
    tf.keras.layers.Dense(p1, activation = 'relu', use_bias = False),
    tf.keras.layers.Dense(1, use_bias = False)
])
model_tf.compile(
    loss = 'mse',
    optimizer= tf.keras.optimizers.SGD(learning_rate=1e-14),
    #optimizer = 'RMSprop',
    #optimizer = 'Adam',
    #optimizer = opt,
)

model_tf.fit(
    X_train,
    y_train,
    batch_size = 2000,
    epochs = 400,
    verbose = 0,
    validation_data=(X_test, y_test)
)
```

In [50]:
p = X_train.shape[1]
# W1: p by p1
model_tf = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape = (p, )), # this defines the number of rows of W1
    tf.keras.layers.Dense(p1, activation = 'relu', use_bias = False), # u1 = X@W1, h1 = relu(u1)
    tf.keras.layers.Dense(1, use_bias = False) #  u2 = h1@W2 (1 is the 2nd dim of y.)
])

In [52]:
# In our simple NN, we have trained W1, W2
# this is the same in model_tf
print(W1.shape[0]*W1.shape[1])
print(W2.shape[0]*W2.shape[1])

155
5


In [53]:
model_tf.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 5)              │           155 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160 (640.00 B)

 Trainable params: 160 (640.00 B)

 Non-trainable params: 0 (0.00 B)

In [54]:
X_train.shape

(2197, 31)

In [56]:
model_tf.compile(
    loss = 'mse',
    optimizer= tf.keras.optimizers.SGD(learning_rate=1e-14),
    #optimizer = 'RMSprop',
    #optimizer = 'Adam',
    #optimizer = opt,
)

model_tf.fit(
    X_train,
    y_train,
    batch_size = 110, # this is like in simple_NN(), X_train will only have 110 many rows
    # tensorflow will split our X_train into about 2197/110 \approx 20 portions
    epochs = 400, # 1 epoch means the package will run all 20 portions (go through the whole X_train ONCE)
    verbose = 0,
    validation_data=(X_test, y_test)
)

In [57]:
root_mean_squared_error(model_tf.predict(X_test), y_test)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 730us/step


200615.453125

In [ ]:
# to be continued